# phase_2 — generate detector boxes for ALL MIMIC (for phase_3)

Runs the trained 29-region YOLO (`best.pt`, pulled from Drive) over every MIMIC 448 image and
writes `predictions.jsonl`, then (if the phase_3 manifest is provided) the phase_3-ready
`boxes_det.npy` / `present_mask_det.npy`. Everything is pushed to Drive via the SAME OAuth-token
logic as the training notebook (service accounts can't upload — see the auth cell).

phase_3 consumes the boxes with `config.BOX_SOURCE="detector"` (default) — M3 then sees the same
box source at train and at launch (concern B1).

## CONFIG — edit these

In [ ]:
import os
PHASE2_SRC = "/kaggle/working/repo/phase_2"
IMAGES     = "/kaggle/input/datasets/nguynnghin/mimic-cxr-448/mimic-cxr-448"  # 448 images (stem == image_id)

# best.pt on Drive (synced by the training notebook). Adjust if your run name differs.
WEIGHTS_REMOTE = "dhint:phase2_runs/det29/weights/best.pt"

# OPTIONAL: phase_3 manifest (from phase_3/labels.py) uploaded as a Kaggle dataset. If set, the
# notebook also builds boxes_det.npy aligned to it. Leave "" to only produce predictions.jsonl
# (then run phase_3/boxes_from_pred.py locally against your manifest).
MANIFEST   = "/kaggle/input/datasets/nguynnghin/m3-labels/manifest.jsonl"   # or ""

INFER_CONF = 0.25
INFER_IMGSZ = 448
BATCH      = 32

# ===== Drive (YOUR OAuth token — see auth section) =====
DRIVE_FOLDER_ID = "1c6AJ3fjsL449kiMK4xiXfnzwzA4gIo0O"   # CHEX-DATA folder id
RCLONE_REMOTE   = "dhint:m3_boxes"                      # = CHEX-DATA/m3_boxes

for k, v in dict(PHASE2_SRC=PHASE2_SRC, IMAGES=IMAGES, WEIGHTS_REMOTE=WEIGHTS_REMOTE,
                 MANIFEST=MANIFEST, INFER_CONF=INFER_CONF, INFER_IMGSZ=INFER_IMGSZ, BATCH=BATCH,
                 DRIVE_FOLDER_ID=DRIVE_FOLDER_ID, RCLONE_REMOTE=RCLONE_REMOTE).items():
    os.environ[k] = str(v)
print("config | IMAGES =", IMAGES, "| weights =", WEIGHTS_REMOTE)


In [ ]:
# --- code from GitHub. Internet ON. ---
!rm -rf /kaggle/working/repo && git clone -q https://github.com/hiennguyendang/phase_2_3_4_5 /kaggle/working/repo
!ls /kaggle/working/repo/phase_2 | head


## 1 — install rclone + auth via YOUR Google account (OAuth token)
Service accounts 403 on upload to My Drive; authenticate as you. One-time: `rclone authorize "drive"`
on a machine with a browser → put the `{...}` token in Kaggle Secret **`GDRIVE_TOKEN`**.

In [ ]:
%%bash
set -e
if ! command -v rclone >/dev/null 2>&1; then
  cd /kaggle/working && curl -sLO https://downloads.rclone.org/rclone-current-linux-amd64.zip
  unzip -q -o rclone-current-linux-amd64.zip && cp rclone-*-linux-amd64/rclone /usr/local/bin/ && chmod +x /usr/local/bin/rclone
fi
rclone version | head -1


In [ ]:
import os
SYNC_OK = "0"
try:
    from kaggle_secrets import UserSecretsClient
    sec = UserSecretsClient()
    token = sec.get_secret("GDRIVE_TOKEN").strip()
    os.environ["RCLONE_CONFIG_DHINT_TYPE"] = "drive"
    os.environ["RCLONE_CONFIG_DHINT_TOKEN"] = token
    os.environ["RCLONE_CONFIG_DHINT_SCOPE"] = "drive"
    os.environ["RCLONE_CONFIG_DHINT_ROOT_FOLDER_ID"] = os.environ["DRIVE_FOLDER_ID"]
    os.environ.pop("RCLONE_CONFIG_DHINT_SERVICE_ACCOUNT_FILE", None)
    for k_sec, k_env in [("GDRIVE_CLIENT_ID","RCLONE_CONFIG_DHINT_CLIENT_ID"),
                         ("GDRIVE_CLIENT_SECRET","RCLONE_CONFIG_DHINT_CLIENT_SECRET")]:
        try: os.environ[k_env] = sec.get_secret(k_sec).strip()
        except Exception: pass
    remote = os.environ["RCLONE_REMOTE"]
    if os.system('rclone mkdir "%s"' % remote) == 0 and os.system('echo ok | rclone rcat "%s/_write_test.txt"' % remote) == 0:
        SYNC_OK = "1"; os.system('rclone delete "%s/_write_test.txt" 2>/dev/null' % remote)
        print("remote OK (OAuth, write verified) ->", remote)
    else:
        print("[WARN] Drive write FAILED -> check GDRIVE_TOKEN + DRIVE_FOLDER_ID")
except Exception as e:
    print("[WARN] no GDRIVE_TOKEN -> results stay in /kaggle/working only:", e)
os.environ["SYNC_OK"] = SYNC_OK
print("SYNC_OK =", SYNC_OK)


## 2 — pull best.pt from Drive + install deps

In [ ]:
!rm -rf /kaggle/working/phase_2 && cp -r "$PHASE2_SRC" /kaggle/working/phase_2
%cd /kaggle/working/phase_2
!pip install -q "ultralytics==8.4.80"
!mkdir -p /kaggle/working/weights
!rclone copy "$WEIGHTS_REMOTE" /kaggle/working/weights --quiet
!ls -lh /kaggle/working/weights || echo "[WARN] best.pt not pulled — check WEIGHTS_REMOTE"


## 3 — infer over all MIMIC → predictions.jsonl
One pass (no resume); ~220k imgs on 1×T4 fit a session. If it dies, just Run-All again. The combined
`predictions.jsonl` is pushed to Drive at the end.

In [ ]:
import os
# build the command in Python (robust; avoids IPython "!...\" continuation quirks)
cmd = ('python infer_yolo.py --weights /kaggle/working/weights/best.pt --source "%s" '
       '--out /kaggle/working/pred --no-per-image --conf %s --imgsz %s --batch %s') % (
       os.environ["IMAGES"], os.environ["INFER_CONF"], os.environ["INFER_IMGSZ"], os.environ["BATCH"])
print(cmd)
os.system(cmd)
os.system("ls -lh /kaggle/working/pred/predictions.jsonl")
if os.environ.get("SYNC_OK") == "1":
    os.system('rclone copy /kaggle/working/pred/predictions.jsonl "%s" --quiet' % os.environ["RCLONE_REMOTE"])
    print("predictions.jsonl -> Drive")


## 4 — (optional) build phase_3 detector boxes
Needs the phase_3 manifest (from `phase_3/labels.py`). Produces `boxes_det.npy` +
`present_mask_det.npy` aligned to it; pushed to Drive. Skip if MANIFEST="" and run
`phase_3/boxes_from_pred.py` locally instead.

In [ ]:
import os
if os.environ.get("MANIFEST") and os.path.exists(os.environ["MANIFEST"]):
    cmd = ('python /kaggle/working/repo/phase_3/boxes_from_pred.py '
           '--pred /kaggle/working/pred/predictions.jsonl --manifest "%s" '
           '--out-dir /kaggle/working/m3_boxes') % os.environ["MANIFEST"]
    print(cmd)
    os.system(cmd)
    if os.environ.get("SYNC_OK") == "1":
        os.system('rclone copy /kaggle/working/m3_boxes "%s" --quiet' % os.environ["RCLONE_REMOTE"])
        print("boxes_det.npy + present_mask_det.npy -> Drive")
    os.system("ls -lh /kaggle/working/m3_boxes")
else:
    print("MANIFEST not found -> only predictions.jsonl produced. "
          "Run phase_3/boxes_from_pred.py locally against your manifest.")
